# Script 6: Classify Articles by Narrative Intent
This script classifies articles into narrative intent categories: "inform", "teach", "persuade", "connect", or "explore".
We use the article `form_factor` feature we extracted earlier and apply Fenic's `semantic.classify` function to determine the author's primary relationship with the reader. 
Few-shot examples are provided to guide the model, including borderline cases with definitive stances on tricky classifications like technical tutorials with philosophical elements or data analysis with policy recommendations.

In [ ]:
import fenic as fc
from dotenv import load_dotenv

load_dotenv()

fc.configure_logging()

config = fc.SessionConfig(
        app_name="medium_curation",
        semantic=fc.SemanticConfig(
            language_models={
                "flash": fc.GoogleGLAModelConfig(
                    model_name="gemini-2.0-flash",
                    rpm=2000,
                    tpm=4_000_000,
                ),
            },
        ),
    )

session = fc.Session.get_or_create(config)

In [13]:
source = session.table("with_features").select("url", "title", "text", "form_factor")

## Step 1: Define handcurated few-shot classification examples

In [14]:
example_collection = fc.ClassifyExampleCollection()

# ----------------------------------
# INFORM — "Here's what you need to know"
# ----------------------------------
inform_examples = [
    # Clear inform examples
    """
    The article presents a comprehensive overview of transformer architecture developments in 2025,
    documenting the shift from attention-based models to hybrid approaches. It systematically covers
    key innovations including sparse attention mechanisms, mixture-of-experts scaling, and memory-efficient
    training techniques. The author provides detailed statistics on model performance improvements and
    computational efficiency gains, supported by benchmarks from leading research institutions.
    The piece concludes with a summary of current limitations and ongoing research directions in the field.
    """,
    """
    The article examines the current state of quantum computing hardware, tracing the evolution from
    50-qubit systems to today's 1000+ qubit processors. It methodically explains the different technological
    approaches including superconducting circuits, trapped ions, and photonic systems, providing performance
    comparisons and error rates for each. The author includes recent breakthrough announcements from IBM,
    Google, and emerging startups, along with realistic timelines for achieving quantum advantage in various applications.
    """,

    # Borderline inform example
    """
    The article analyzes the regulatory landscape surrounding AI development across major economies,
    documenting recent policy changes in the EU, US, and China. It presents a factual comparison of
    different regulatory approaches, highlighting key differences in data protection requirements,
    algorithmic auditing mandates, and liability frameworks. While noting potential challenges with
    implementation, the author maintains an objective tone throughout, focusing on documented policy
    positions rather than advocating for specific approaches.
    """
]

# ----------------------------------
# TEACH — "Let me show you how to do this"
# ----------------------------------
teach_examples = [
    # Clear teach examples
    """
    The article provides a complete tutorial on implementing federated learning for mobile applications,
    starting with environment setup and dependency installation. It walks through creating client-side
    model training loops, implementing secure aggregation protocols, and handling communication with
    central servers. Each section includes detailed code examples in Python and Flutter, with explanations
    of key concepts like differential privacy and byzantine fault tolerance. The guide concludes with
    deployment strategies and monitoring techniques for production environments.
    """,
    """
    The article offers a step-by-step guide to building custom GPT applications using the latest
    OpenAI API features. It begins with authentication setup and API key configuration, then demonstrates
    prompt engineering techniques for specific use cases. The tutorial includes practical examples of
    function calling, streaming responses, and error handling, with complete code snippets for common
    scenarios like document analysis and conversation systems. The author provides optimization tips and
    best practices for cost-effective implementation.
    """,

    # Borderline teach example
    """
    The article teaches readers how to think about AI system design from a product perspective, introducing
    frameworks for evaluating AI capabilities against user needs. It guides readers through a structured
    approach to AI product development, from problem definition to solution architecture. While including
    some code examples for evaluation metrics, the focus is on developing strategic thinking skills rather
    than implementing specific technologies. The piece includes exercises and reflection questions to help
    readers apply these frameworks to their own projects.
    """
]

# ----------------------------------
# PERSUADE — "You should think/act differently"
# ----------------------------------
persuade_examples = [
    # Clear persuade examples
    """
    The article makes a compelling case for why software engineers must embrace AI pair programming or
    risk career obsolescence. It argues that resistance to AI tools represents a fundamental misunderstanding
    of technological evolution, drawing parallels to historical shifts like the transition from assembly to
    high-level languages. The author challenges common objections about AI reliability and creativity,
    presenting evidence that AI-assisted developers consistently outperform traditional approaches.
    The piece concludes with urgent recommendations for skill development and a call to action for
    engineering teams to adopt AI workflows immediately.
    """,
    """
    The article argues that current AI safety research is fundamentally misdirected, advocating for a
    complete paradigm shift toward interpretability over alignment. The author contends that focusing on
    controlling AI behavior misses the critical need to understand AI decision-making processes.
    Through analysis of recent AI failures and philosophical arguments about consciousness, the piece
    builds a case that interpretability research deserves the majority of safety funding. The article
    concludes with specific policy recommendations and a call for researchers to redirect their efforts.
    """,

    # Borderline persuade example
    """
    The article presents a thoughtful argument for reconsidering remote work policies in the age of
    AI collaboration tools. The author suggests that traditional assumptions about in-person productivity
    may no longer apply given advances in virtual collaboration and AI-mediated communication.
    While acknowledging valid concerns about team cohesion and spontaneous innovation, the piece
    advocates for hybrid approaches that leverage AI tools to bridge physical distances. The article
    concludes with gentle recommendations for organizations to experiment with AI-enhanced remote workflows.
    """
]

# ----------------------------------
# CONNECT — "Let me share this experience with you"
# ----------------------------------
connect_examples = [
    # Clear connect examples
    """
    The article chronicles the author's emotional journey through their first AI startup failure, sharing
    intimate details about the crushing disappointment of a failed product launch and subsequent team
    dissolution. The narrative unfolds through personal anecdotes about sleepless nights debugging models,
    difficult conversations with investors, and the gradual realization that their core assumptions were wrong.
    The author vulnerably describes their struggle with imposter syndrome and the slow process of rebuilding
    confidence, concluding with hard-won insights about resilience and the importance of community support
    in the tech industry.
    """,
    """
    The article shares the author's experience as a non-technical founder learning to work with AI engineers,
    detailing the cultural and communication barriers encountered along the way. Through candid stories
    about misunderstood requirements, failed project timelines, and breakthrough moments of mutual understanding,
    the piece reveals the human side of technical collaboration. The author reflects on moments of feeling
    excluded from technical discussions and the gradual development of enough AI literacy to contribute
    meaningfully to product decisions.
    """,

    # Borderline connect example
    """
    The article recounts the author's six-month experiment living entirely on AI-generated meal plans,
    shopping lists, and recipes. The narrative follows their daily experiences with AI nutritionist apps,
    documenting both surprising successes and frustrating failures in meal planning automation.
    While sharing personal details about changed eating habits and health impacts, the author weaves in
    broader observations about AI's current limitations in understanding individual preferences and
    cultural contexts around food.
    """
]

# ----------------------------------
# EXPLORE — "Let's think about this together"
# ----------------------------------
explore_examples = [
    # Clear explore examples
    """
    The article contemplates the profound implications of AI systems that can modify their own code,
    raising fundamental questions about the nature of software, consciousness, and control. The author
    guides readers through thought experiments about recursive self-improvement, wondering whether
    we're approaching a threshold where traditional programming paradigms become obsolete. Rather than
    providing definitive answers, the piece invites readers to consider multiple perspectives on AI agency,
    the philosophy of mind, and what it might mean for artificial systems to truly understand themselves.
    """,
    """
    The article explores the curious phenomenon of AI models exhibiting emergent behaviors that seem
    to transcend their training, questioning what this reveals about intelligence and learning.
    The author examines recent examples of unexpected AI capabilities, wondering whether these represent
    genuine understanding or sophisticated pattern matching at scales we don't fully comprehend.
    The piece invites readers to join a philosophical investigation into the boundaries between mimicry
    and genuine cognition, leaving space for multiple interpretations and ongoing questions.
    """,

    # Borderline explore example
    """
    The article examines the puzzling question of why AI models sometimes perform better on complex
    tasks than simple ones, exploring various hypotheses about emergent reasoning capabilities.
    The author considers multiple explanations including training data distribution, computational scaling
    effects, and the possibility of genuine reasoning emergence. While presenting evidence for different
    theories, the piece maintains an open-ended inquiry approach, acknowledging the fundamental uncertainty
    around AI cognition and inviting readers to consider what these patterns might reveal about intelligence itself.
    """
]


In [15]:
for example in inform_examples:
    example_collection.create_example(fc.ClassifyExample(input=example, output="inform"))

for example in teach_examples:
    example_collection.create_example(fc.ClassifyExample(input=example, output="teach"))

for example in persuade_examples:
    example_collection.create_example(fc.ClassifyExample(input=example, output="persuade"))

for example in connect_examples:
    example_collection.create_example(fc.ClassifyExample(input=example, output="connect"))

for example in explore_examples:
    example_collection.create_example(fc.ClassifyExample(input=example, output="explore"))

## Step 2: Run few-shot classification

In [16]:
with_narrative_intent_label = source.with_column(
    "narrative_intent_label",
    fc.semantic.classify(
        "form_factor",
        labels=[
            "inform",
            "teach",
            "persuade",
            "connect",
            "explore"
        ],
        examples=example_collection
    )
).cache()

In [ ]:
with_narrative_intent_label.group_by("narrative_intent_label") \
   .agg(fc.count("*").alias("count")) \
   .sort(fc.col("count").desc()) \
   .show()

In [ ]:
with_narrative_intent_label.select("narrative_intent_label", "form_factor").show(60)

## Step 3: Save Results

In [ ]:
with_narrative_intent_label.write.save_as_table("with_narrative_intent_label", mode="overwrite")

In [23]:
session.stop()